# LatentFT lab: the rate axis, bin by bin

LatentFT (Wang & Huang, *Latent Fourier Transform*, ICLR 2026) encodes about 6 s of audio into a latent of 80 channels
that change 86 times a second, takes the Fourier transform of each channel **along time**, and lets you choose which
of the 513 resulting *rate bins* (0 to 43 Hz, 0.084 Hz apart) the decoder is given before it turns the latent back
into sound. Slow rates carry bars and phrases; fast rates carry 16th notes, rolls and attacks. Whatever you remove is
not silenced: the decoder **regenerates** it, consistent with what you kept. It is a generative EQ on musical time.

The web page at https://latentft.csail.mit.edu works with the paper's 8 log bands. This notebook has the whole axis:
Hz ranges, single bins, a gain per bin, blend weights between two clips, several models, and the raw latent itself.

**Where to run it.** The easy way is the hosted notebook at https://latentft.csail.mit.edu/jupyter/ : sign in with a name and your class token, and this notebook is already open with everything installed. Otherwise:

| mode | what you need | where the GPU is |
|---|---|---|
| `hub` | your class token | the lab's GPU behind https://latentft.csail.mit.edu (shared: be modest) |
| `local` | your own `latentft_server.py` (e.g. an Engaging job, see `engaging/README.md`) and an ssh tunnel | your job |
| `in_process` | the model repo and weights on the machine this kernel runs on (Engaging OnDemand Jupyter with a GPU) | this kernel |

The model weights are Mason Wang's and are not public; in `hub` mode they never leave the lab.

**Bands** (the paper's log split of the rate axis, musical names at 120 BPM): 1 = slower than a bar, plus each
channel's mean (overall timbre and level) · 2 = the bar · 3 = half notes · 4 = the beat · 5 = 8ths · 6 = 16ths ·
7 = 32nds · 8 = attacks and micro-timing. At other tempos the names shift; the Hz do not.

In [ ]:
# the client is a small package served by the class site; the hosted notebook has it already
import importlib.util, subprocess, sys
if importlib.util.find_spec("latentft_client") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--extra-index-url", "https://latentft.csail.mit.edu/pip/simple",
                    "latentft-lab[viz]"], check=False)
if importlib.util.find_spec("latentft_client") is None:          # no pip? the two files directly
    import pathlib, urllib.request
    for fn in ("latentft_client.py", "latentft_viz.py"):
        urllib.request.urlretrieve(f"https://latentft.csail.mit.edu/notebook/{fn}", fn)


In [ ]:
import getpass, numpy as np, matplotlib.pyplot as plt, soundfile as sf, librosa
from IPython.display import Audio, display
from latentft_client import LatentFT, keep, lowpass, highpass, soft_lowpass, ALL, SR, WINDOW_S, BIN_HZ
from latentft_viz import show, show_all, describe, onset_peaks, soft_band, soft_octave, xfade_sweep, sweep_figure

MODE = "hub"          # "hub" | "local" | "in_process"
if MODE == "hub":
    try:
        lf = LatentFT()                                          # the hosted notebook has your token on file
    except RuntimeError:
        lf = LatentFT(token=getpass.getpass("class token: "))
elif MODE == "local":
    lf = LatentFT(hub="http://127.0.0.1:8765", token=None)      # the tunnel to your latentft_server.py
else:
    REPO = "/path/to/latentft-post-training"                     # Engaging: where you cloned it
    sys.path += [f"{REPO}/demo", f"{REPO}/colab"]
    lf = LatentFT.in_process({"mono": "/path/to/latentft_mlp_5s_ema_fp32.pt"})

models = lf.models()
for m in models:
    print(f'{m["name"]:<7} {"online" if m.get("online") else "offline":<8} {m.get("label", "")}')
MODEL = next((m["name"] for m in models if m.get("online")), "mono")
print("using", MODEL)

## 1 · Pick a clip
A loop from the class library (Freesound Loop Dataset; every loop keeps its creator's Creative Commons licence), or any
file of your own with `a, sr = sf.read("my.wav")`. One model window is 5.944 s; the client sends exactly the
selection, so uploads stay small.

In [ ]:
try:
    lib = lf.library()
    for l in lib[:12]:
        print(f'{l["id"]:>7}  {l["group"]:<6} {l["genre"]:<16} {l["bpm"]:>5.0f} BPM  {l["title"][:46]}  ({l["credit"]}, {l["license"]})')
    A_ID = lib[0]["id"]                      # change me
    a, sr = lf.loop(A_ID)
    BPM_A = next(l["bpm"] for l in lib if l["id"] == A_ID)
except Exception as e:                       # no library in local / in_process mode: bring your own file
    print("no library here:", e)
    a, sr = sf.read("my.wav", dtype="float32", always_2d=True); BPM_A = None
display(Audio(a.T, rate=sr))

## 2 · Look at the rate axis
`encode` returns the latent of one window (80 channels x 512 frames for the mono model, 160 x 512 for stereo).
Its rate spectrum is the Fourier transform of each channel along time. This is the picture every mask acts on.

In [ ]:
enc = lf.encode(a, sr, model=MODEL)
print(enc)
Z = enc.spectrum()                                   # (C, 513) complex
P = 20 * np.log10(np.abs(Z) + 1e-9)
f = enc.freqs
fig, ax = plt.subplots(figsize=(11, 3.2))
im = ax.pcolormesh(f[1:], np.arange(P.shape[0]), P[:, 1:] - P[:, 1:].max(), vmin=-60, vmax=0, shading="auto", cmap="magma")
ax.set_xscale("log"); ax.set_xlabel("rate (Hz)"); ax.set_ylabel("latent channel"); ax.set_title("the latent's rate spectrum (dB re max; bin 0, the mean, left out)")
for b in enc.model_info["bands"][1:]:
    ax.axvline(b["lo_hz"], color="w", lw=0.5)
if BPM_A:
    for name, hz in (("bar", BPM_A / 240), ("beat", BPM_A / 60), ("8th", BPM_A / 30), ("16th", BPM_A / 15)):
        ax.axvline(hz, color="c", lw=1, ls="--"); ax.text(hz, P.shape[0] - 6, name, color="c", fontsize=8)
plt.colorbar(im, ax=ax, pad=0.01); plt.tight_layout()

## 3 · The paper's bands first
Keep bands 1..k (a low-pass on the rate axis) and listen as k grows. All renders of one call share one starting
noise, so they line up in time. The response's `keep` says what was applied, in bins and in Hz.

In [ ]:
res = lf.render(a, sr, [{"a": ALL[:k]} for k in range(1, 9)], model=MODEL, seconds=WINDOW_S, bpm_a=BPM_A, return_latents=True)
marks = onset_peaks(a, sr)
print("onset-rate peaks:", ", ".join(f"{name} {hz:.2f} Hz" for hz, name in marks))
print(res, "| clip tempo:", res.clip_a["bpm"], "BPM, beat in band", res.clip_a["beat_band"])
print("what the model hears (mel + vocoder, no model):"); display(Audio(res.ref("a").T, rate=SR))
show_all(res, marks=marks)        # per render: what the mask does, a player, and the five panels


### The ladder at a glance
An overview, one row per render: the output's spectrogram and the model's own view of it (latent channel × rate,
the kept bins outlined), and a slider that steps through the renders one at a time with sound (no animation).
`gallery(...)` and `pick(...)` work on any result in this notebook.

In [ ]:
import librosa.display

def gallery(res, which="a", max_rows=10, title=None):
    """One row per render: spectrogram (log-frequency mel, dB) and the model's picture (latent channel x rate, dB)."""
    n = min(len(res), max_rows)
    hz = np.array(res.rate_hz)
    fig, axes = plt.subplots(n + 1, 2, figsize=(13, 1.5 * (n + 1)), gridspec_kw={"width_ratios": [2.2, 1]}, squeeze=False)
    def spec(ax, y, lab):
        S = librosa.power_to_db(librosa.feature.melspectrogram(y=y.mean(1), sr=SR, n_mels=96, hop_length=256, fmax=11025), ref=np.max)
        librosa.display.specshow(S, sr=SR, hop_length=256, y_axis="mel", fmax=11025, ax=ax, cmap="magma", vmin=-70, vmax=0)
        ax.set_ylabel(lab, rotation=0, ha="right", va="center", fontsize=8); ax.set_yticks([]); ax.set_xlabel("")
    def rate(ax, rows, keep):
        if rows is None:
            ax.axis("off"); return
        ax.pcolormesh(hz, np.arange(len(rows)), np.array(rows), vmin=-60, vmax=0, shading="auto", cmap="magma")
        ax.set_xscale("log"); ax.set_yticks([])
        k = keep or {}
        for lo, hi in (k.get("hz") or k.get("support_hz") or []):          # kept bins (soft: every bin with any gain)
            ax.axvspan(max(lo, hz[0]), max(hi, hz[0] * 1.05), fill=False, edgecolor="w", lw=1, ls="--" if k.get("soft") else "-")
        if k.get("soft"):
            for lo, hi in (k.get("full_hz") or []):                        # soft: the part kept at full gain, solid
                ax.axvspan(max(lo, hz[0]), max(hi, hz[0] * 1.05), fill=False, edgecolor="w", lw=1)
    clip = res.clip_a if which == "a" else res.clip_b
    spec(axes[0, 0], res.ref(which), "what the model hears"); rate(axes[0, 1], clip.get("rate_db"), None)
    for i in range(n):
        spec(axes[i + 1, 0], res.audio(i), res.label(i)); rate(axes[i + 1, 1], res.renders[i].get("rate_db"), res.keep(i, which))
    axes[0, 0].set_title(title or "spectrogram (log frequency)", fontsize=9)
    axes[0, 1].set_title("the model's picture: latent channel × rate (Hz); kept bins outlined", fontsize=9)
    for ax in axes[:-1].ravel():
        ax.set_xticks([])
    axes[-1, 0].set_xlabel("time (s)"); axes[-1, 1].set_xlabel("rate (Hz)")
    plt.tight_layout()
    return fig

def pick(res):
    """A stepped slider: one render at a time, its label and its sound."""
    try:
        from ipywidgets import interact, IntSlider
    except ImportError:
        print("ipywidgets is not installed; use res.audio(i)"); return
    @interact(i=IntSlider(value=0, min=0, max=len(res) - 1, step=1, continuous_update=False, description="render"))
    def _show(i):
        print(res.label(i)); display(Audio(res.audio(i).T, rate=SR))

gallery(res, title="low-pass ladder: keep bands 1..k");
pick(res)

## 4 · Any bins you like
`keep(hz=...)`, `keep(bins=...)` and `keep(mask=...)` address the axis directly. Below: only the octave around the
clip's beat rate; a one-bin notch at the bar rate; a brick-wall low-pass at 2.5 Hz; and the same cutoff with a
raised-cosine edge. The last one is a *soft* mask: a gain between 0 and 1 on some bins.

**What a soft mask means.** The model multiplies each rate bin of the latent's spectrum by the mask value. With 0 or 1
that is exactly what it was trained on: a zeroed bin means "no information here, regenerate". A gain of 0.3 hands the
decoder that rate's pattern at 30 % amplitude, which it never saw in training, so it is **not** "30 % kept". Useful
as a smoother band edge, and as an experiment (at what gain does the decoder stop locking onto the beat?), but for a
controllable partial effect use blend `weights` (section 5).

In [ ]:
bpm = res.clip_a["bpm"] or 120.0
beat, bar = bpm / 60, bpm / 240
print(f"tempo {bpm:.0f} BPM -> bar {bar:.3f} Hz, beat {beat:.3f} Hz, 8ths {2 * beat:.2f} Hz, 16ths {4 * beat:.2f} Hz; "
      f"one rate bin is {BIN_HZ:.4f} Hz, the axis ends at {512 * BIN_HZ:.2f} Hz")
renders = [
    {"a": keep(hz=[[beat / 1.41, beat * 1.41]])},                       # the octave around the beat
    {"a": keep(mask=np.r_[np.ones(513)] * (np.abs(enc.freqs - bar) > BIN_HZ * 0.5))},   # notch: drop only the bar bin
    {"a": lowpass(2.5)},
    {"a": soft_lowpass(2.5, width_hz=1.0)},
    {"a": keep(bins=[[0, 15], [128, 512]])},                            # slow + very fast, nothing in between
]
res4 = lf.render(a, sr, renders, model=MODEL, seconds=WINDOW_S, bpm_a=BPM_A, return_latents=True)

def onset_env(y):
    return librosa.onset.onset_strength(y=y.mean(1), sr=SR)
def rhythm_match(y, ref):
    o, r = onset_env(y), onset_env(ref); n = min(len(o), len(r))
    return float(np.corrcoef(o[:n], r[:n])[0, 1])

ref = res4.ref("a")
for i in range(len(res4)):
    k = res4.keep(i)
    hz = k["hz"] if k["hz"] is not None else k["support_hz"]
    print(f"{i}: {res4.label(i)}")
    print(f"     Hz kept: {hz}" + (f"   full gain: {k['full_hz']}" if k["soft"] else "") + f"   bins: {k['bins_kept']}, dc={k['dc']}, "
          f"rhythm like the original: {rhythm_match(res4.audio(i), ref):+.2f}")
    display(Audio(res4.audio(i).T, rate=SR))
gallery(res4, title="per-bin specs");



### Each render, with its pictures
`show_all` presents every render the way the listening pages do: a sentence saying what the mask does, a player, and
five panels: the **mask** as a graph over the rate axis (1 = that speed of change is kept from the clip, 0 = erased and
regenerated, in between = kept faintly); the clip's **latent** (channels over the first 5.94 s, red positive, blue
negative); the latent **the decoder was given** (masked along time); the **output's latent**, re-encoded; and the
output's spectrogram. Dotted lines mark the clip's own onset rates. Needs `return_latents=True` on the render.

In [ ]:
show_all(res4, marks=marks)



### A zone gliding across the axis
One soft one-octave zone at several centres, each a full render from the same noise, then crossfaded in order so the
zone moves while the music plays. Use a longer selection here (up to 30 s): a 10-s pass is too quick to hear the
motion. `band=True` keeps the zone, `band=False` erases it.

In [ ]:

SWEEP_S = min(30.0, len(a) / sr)                     # the selection the glide uses (longer is better, 30 s max)
centres = np.geomspace(0.4, 16.0, 8)
band = True                                          # True: keep the moving octave; False: erase it
glide = [{"a": keep(mask=(soft_octave(fc) if band else 1 - soft_octave(fc)))} for fc in centres]
res_g = lf.render(a, sr, glide, model=MODEL, seconds=SWEEP_S, bpm_a=BPM_A)
print(res_g, "| centres:", np.round(centres, 2), "Hz")
renders = [res_g.audio(i) for i in range(len(res_g))]
for name, passes in (("up", [True]), ("up then down", [True, False])):
    piece = xfade_sweep(renders, passes)              # repeats=2 plays the selection twice per pass
    print(f"{name}: {len(piece) / SR:.0f} s"); display(Audio(piece.T, rate=SR))
    sweep_figure(piece, centres, passes, marks=marks, words=f"the {'kept' if band else 'erased'} octave moves {name} from {centres[0]:.1f} to {centres[-1]:.1f} Hz", title=f"glide {name}", band=band, model=MODEL);


## 5 · Two clips: blend, with weights
The paper's blend: clip A's latent masked to its bins, clip B's to its bins, and the sampler follows both at once.
`weights` tilts the guidance (default equal). Take a drum loop for A and something tonal for B. Measured on two
songs, the rhythm follows whichever clip supplies the 8th notes and faster (bands 5-8); does that hold for your pair,
and does the hand-over move when the weights do?

In [ ]:
try:
    B_ID = next(l["id"] for l in lib if l["group"] == "tonal")     # change me
    b, sr_b = lf.loop(B_ID); BPM_B = next(l["bpm"] for l in lib if l["id"] == B_ID)
except Exception:
    b, sr_b = sf.read("other.wav", dtype="float32", always_2d=True); BPM_B = None
display(Audio(b.T, rate=sr_b))

cut = 2.7                                                            # Hz: A below, B above
sweep = [{"a": keep(hz=[[0, cut]]), "b": keep(hz=[[cut, 43.07]]), "weights": [w, round(1 - w, 2)]} for w in (0.9, 0.7, 0.5, 0.3, 0.1)]
res5 = lf.render(a, sr, sweep, b=b, sr_b=sr_b, model=MODEL, seconds=WINDOW_S, bpm_a=BPM_A, bpm_b=BPM_B)
ra, rb = res5.ref("a"), res5.ref("b")
for i in range(len(res5)):
    y = res5.audio(i)
    print(f"{res5.label(i):<60} rhythm like A {rhythm_match(y, ra):+.2f}  like B {rhythm_match(y, rb):+.2f}")
    display(Audio(y.T, rate=SR))
gallery(res5, which="a", title="weights sweep: A slow / B fast (pictures vs clip A)");


In [ ]:
# the crossover ladder by Hz instead of by band: where along the axis does the rhythm change hands?
cuts = np.geomspace(0.5, 20, 7)
ladder = [{"a": keep(hz=[[0, c]]), "b": keep(hz=[[c, 43.07]])} for c in cuts]
res5b = lf.render(a, sr, ladder, b=b, sr_b=sr_b, model=MODEL, seconds=WINDOW_S, bpm_a=BPM_A, bpm_b=BPM_B)
for c, i in zip(cuts, range(len(res5b))):
    y = res5b.audio(i)
    print(f"A below {c:5.2f} Hz, B above:   like A {rhythm_match(y, ra):+.2f}   like B {rhythm_match(y, rb):+.2f}")

## 6 · The latent itself
`encode` gives you the latent; `decode` renders latents you changed. Everything in between is numpy, so a mask can
depend on the data (keep only the bins where this clip is strong), pick bins from two clips by comparing them, or
move energy along the axis. Latents must keep the model's shape (C x 512); `Encoded.spectrum()` and
`Encoded.from_spectrum()` go to the rate domain and back the way the service does.

In [ ]:
ea, eb = lf.encode(a, sr, model=MODEL), lf.encode(b, sr_b, model=MODEL)
Za, Zb = ea.spectrum(), eb.spectrum()

# (i) data-dependent mask: keep the bins where A's energy (summed over channels) is above its median; regenerate the rest
pa = np.abs(Za).sum(0); pa[0] = np.inf
strong = (pa >= np.median(pa[1:])).astype(float)
z_strong = ea.masked(strong)

# (ii) per bin, take whichever clip is louder there (A's mean kept): a "winner takes the bin" collage
take_a = np.abs(Za).sum(0) >= np.abs(Zb).sum(0); take_a[0] = True
z_collage = ea.from_spectrum(np.where(take_a[None, :], Za, Zb))

# (iii) everything twice as fast: move each bin k to bin 2k (what lived at the beat rate now lives at the 8th rate)
Z2 = np.zeros_like(Za); Z2[:, 0] = Za[:, 0]
k = np.arange(1, Za.shape[1] // 2); Z2[:, 2 * k] = Za[:, k]
z_double = ea.from_spectrum(Z2)

# (iv) a blend of two raw latents with weights, as decode's own blend
res6 = lf.decode([ea.z, z_strong, z_collage, z_double, {"conds": [ea.z, eb.z], "weights": [0.5, 0.5]}], model=MODEL)
for name, i in zip(["A reconstructed", "A, strong bins only", "collage: louder clip per bin", "A at double rate", "A+B whole, equal weights"], range(len(res6))):
    print(name); display(Audio(res6.audio(i).T, rate=SR))

## 7 · Models
Three checkpoints are served when their worker is online: `mono` (the paper's MLP-encoder model; stereo input is
downmixed), `stereo` (the lab's stereo-v1 fine-tune, left and right as one latent of 160 channels, vocoded per
channel) and `unet` (the paper's U-Net-encoder model, which the paper prefers for isolation; it runs in fp32 and is
slower). Same request, each model; the seed's noise differs per model only because the latent shapes do.

**Stereo has two renderings.** By default each channel goes through the vocoder on its own, so left and right get
unrelated phase: the image is as wide as it can be and uncorrelated, even for the plain reconstruction of a recording
whose channels correlate at +0.6. `stereo_mode="mid"` vocodes the mid once and makes left and right from the model's
per-band level differences, so the channels share a phase like a recording. Try the same render both ways.

In [ ]:
spec = [{"a": keep(hz=[[0, 2.7]])}, {"a": keep(hz=[[2.7, 43.07]])}]
for m in [m["name"] for m in lf.models() if m.get("online")]:
    r = lf.render(a, sr, spec, model=m, seconds=WINDOW_S, bpm_a=BPM_A)
    print(f"{m}: {r.model_info['label']}  ({r.render_s} s on the GPU, {r.model_info['latent_channels']} latent channels)")
    for i in range(len(r)):
        print("  ", r.label(i)); display(Audio(r.audio(i).T, rate=SR))


## 8 · Bulk render, for mapping offline
When the goal is an instrument rather than a listening session, render a whole grid at once and take the files home:
a controller, a SuperCollider or Max patch, or a sampler then maps a knob to the files. `render_many` splits any list
into calls of 48 (the service's limit per call) with the same seed, so every file lines up with every other and
neighbours crossfade cleanly. `save` writes the WAVs plus a `manifest.json` that says, for each file, exactly what it
kept (bins and Hz), so the patch can read the mapping without this notebook. Below: a 2-D grid, zone centre × zone
width, 48 files for 30 s of music.

**Bigger than a notebook cell: a sweep as a job.** The package installs a command, `latentft-sweep`, that renders a
whole sweep in one command, resumably: by default a lowpass sweep (one file per cutoff, 0.25 to 43 Hz), or `--kind band`
for a moving zone; several widths or seeds if you ask, a manifest, optional glide pieces. Run it in a terminal:

```
latentft-sweep --out my_sweep --glides
```

With no audio file it uses Bob Dylan's *The Man in Me* from 2:00; give a path first to use your own (`--start` where to begin).

With [uv](https://docs.astral.sh/uv/) you can skip the install and run it in one line:

```
uvx --python 3.12 --from latentft-lab@latest --index https://latentft.csail.mit.edu/pip/simple latentft-sweep --out my_sweep --glides
```

It prints its progress as it goes. If it stops (you close the terminal, the network drops), run the same command
again: it picks up where it left off.

Files that exist are skipped and the service answers identical requests from its cache, so an interrupted or repeated
job costs nothing. `latentft-sweep --help` lists the options; `--hub http://127.0.0.1:8765` points it at your own server.

The command needs your token like the notebook does. If it says "no token", run it once with `--token YOUR_TOKEN --save-token`; after that it remembers.

In [ ]:

import json, zipfile, pathlib, time
centres = np.geomspace(0.4, 16.0, 16)                 # 16 zone centres (log-spaced)
widths = (1.0, 2.0, 3.0)                              # 3 zone widths in octaves -> 48 renders
grid = [(fc, w) for w in widths for fc in centres]
renders = [{"a": keep(mask=soft_octave(fc, w))} for fc, w in grid]     # keep the zone; use 1 - soft_octave(...) for notches
BULK_S = min(30.0, len(a) / sr)
t0 = time.time()
res_bulk = lf.render_many(a, sr, renders, model=MODEL, seconds=BULK_S, seed=0)
print(res_bulk, f"in {time.time() - t0:.0f} s wall")

out = pathlib.Path(f"bulk_{MODEL}_{int(BULK_S)}s")
paths = res_bulk.save(out, prefix="zone")              # WAVs + manifest.json
m = json.loads((out / "manifest.json").read_text())
for row, (fc, w) in zip(m["renders"], grid):           # add the grid coordinates to the manifest
    row["centre_hz"], row["octaves"] = round(float(fc), 3), w
(out / "manifest.json").write_text(json.dumps(m, indent=1))
with zipfile.ZipFile(f"{out}.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(out.iterdir()):
        z.write(f, f"{out.name}/{f.name}")
print(f"{len(paths)} files -> {out}/  and  {out}.zip  ({pathlib.Path(f'{out}.zip').stat().st_size / 2**20:.0f} MB)")
print("hosted notebook: right-click the zip in the file browser -> Download. manifest.json, first entry:")
print(json.dumps(m["renders"][0], indent=1)[:600])


## 9 · Keep what you made
`Result.save(folder)` writes every render and the references as 22.05-kHz WAV, named after what each kept. The
response dictionaries (`res.raw`) hold everything else: `keep` per render, `rate_db` pictures, tempo estimates.

**Limits and manners.** Up to 48 renders per call, selections up to 30 s (rendered as overlapping 5.944-s windows,
so a 30-s selection costs about 6x one window), 400 window-renders per call. On the hub a byte-identical request is
answered from a shared cache without touching the GPU, so re-running a cell is free; a new seed or a changed mask is
a new render. The class shares one or two GPUs: keep calls modest and `seconds` short while exploring.

The wire format, for other clients (SuperCollider, Max, curl): https://latentft.csail.mit.edu/notebook/api.html

In [ ]:
paths = res4.save("latentft_renders", prefix="section4")
print("\n".join(paths))